# 🎬 Deep-Learning Movie Recommender System

This notebook upgrades the original bag-of-words recommender by replacing
`CountVectorizer` with a pretrained Sentence Transformer.

The model converts each movie's combined metadata into a semantic embedding.
Recommendations are produced by comparing the selected movie embedding with
every other movie embedding.

## Output files

After running the notebook, it creates:

- `movie_dict.pkl` — movie IDs and titles used by Streamlit
- `movie_embeddings.npy` — normalized deep-learning embeddings

The accompanying `app_deep_learning.py` loads these files.

In [ ]:
# Run once if the packages are not installed.
%pip install -U sentence-transformers pandas numpy

In [ ]:
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## 1. Load and merge the TMDB datasets

Keep these files in the same directory as this notebook:

- `tmdb_5000_movies.csv`
- `tmdb_5000_credits.csv`

In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

movies = movies.merge(credits, on="title")
movies = movies[
    ["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]
].copy()

movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

print("Dataset shape:", movies.shape)
movies.head()

## 2. Parse JSON-like columns

TMDB stores genres, keywords, cast, and crew as string representations of
lists of dictionaries. The following functions extract only the values needed
for recommendations.

In [ ]:
def parse_list(value):
    if isinstance(value, str):
        return ast.literal_eval(value)
    return value


def extract_names(value):
    return [
        item["name"]
        for item in parse_list(value)
        if isinstance(item, dict) and item.get("name")
    ]


def extract_top_cast(value, limit=3):
    return extract_names(value)[:limit]


def extract_director(value):
    for item in parse_list(value):
        if isinstance(item, dict) and item.get("job") == "Director":
            return [item["name"]]
    return []

In [ ]:
movies["genres"] = movies["genres"].apply(extract_names)
movies["keywords"] = movies["keywords"].apply(extract_names)
movies["cast"] = movies["cast"].apply(extract_top_cast)
movies["crew"] = movies["crew"].apply(extract_director)
movies["overview"] = movies["overview"].apply(str.split)

movies.head(2)

## 3. Build the combined text feature

Spaces are removed from multi-word names so that people and genres remain
single features, for example:

- `Science Fiction` → `ScienceFiction`
- `Sam Worthington` → `SamWorthington`

In [ ]:
def remove_spaces(items):
    return [str(item).replace(" ", "") for item in items]


for column in ["genres", "keywords", "cast", "crew"]:
    movies[column] = movies[column].apply(remove_spaces)

movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

new_df = movies[["movie_id", "title", "tags"]].copy()
new_df["tags"] = new_df["tags"].apply(lambda values: " ".join(values).lower())
new_df.reset_index(drop=True, inplace=True)

new_df.head()

## 4. Generate deep-learning embeddings

`all-MiniLM-L6-v2` produces a semantic vector for each movie's tags.

`normalize_embeddings=True` makes every embedding unit length. Therefore,
a dot product can be used as cosine similarity.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    new_df["tags"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print("Embedding shape:", embeddings.shape)

## 5. Recommendation function

Instead of storing a large movie-by-movie similarity matrix, this version
calculates one similarity vector only when a movie is selected.

In [ ]:
def recommend(movie_title, number_of_results=5):
    matching_indexes = new_df.index[
        new_df["title"].str.casefold() == movie_title.casefold()
    ].tolist()

    if not matching_indexes:
        return pd.DataFrame(columns=["movie_id", "title", "similarity"])

    movie_index = matching_indexes[0]

    # Embeddings are normalized, so dot product equals cosine similarity.
    scores = embeddings @ embeddings[movie_index]

    ranked_indexes = np.argsort(-scores)

    recommendations = []
    for index in ranked_indexes:
        if index == movie_index:
            continue

        recommendations.append(
            {
                "movie_id": int(new_df.iloc[index]["movie_id"]),
                "title": new_df.iloc[index]["title"],
                "similarity": float(scores[index]),
            }
        )

        if len(recommendations) == number_of_results:
            break

    return pd.DataFrame(recommendations)

In [ ]:
recommend("Spider-Man 3")

## 6. Save the files for Streamlit

These files must be kept beside `app_deep_learning.py`.

In [ ]:
output_directory = Path(".")

with open(output_directory / "movie_dict.pkl", "wb") as file:
    pickle.dump(
        new_df[["movie_id", "title"]].to_dict(),
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

np.save(output_directory / "movie_embeddings.npy", embeddings)

print("Saved movie_dict.pkl")
print("Saved movie_embeddings.npy")

## 7. Run the application

Recommended project structure:

```text
deep-learning-movie-recommender/
├── app_deep_learning.py
├── movie_dict.pkl
├── movie_embeddings.npy
├── deep-learning-movie-recommender.ipynb
├── tmdb_5000_movies.csv
├── tmdb_5000_credits.csv
└── requirements_deep_learning.txt
```

Start Streamlit:

```bash
python -m streamlit run app_deep_learning.py
```